In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("phucthaiv02/butterfly-image-classification")

print("Path to dataset files:", path)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 226M/226M [00:17<00:00, 13.9MB/s] 

Extracting files...


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/phucthaiv02/butterfly-image-classification/versions/3


In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
import os
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])



In [23]:
class Butterflydataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        images_dir = os.path.join(root_dir, "train")
        self.image_paths = sorted(
            os.path.join(images_dir, fname)
            for fname in os.listdir(images_dir)
            if fname.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        labels_dir = os.path.join(root_dir, "Training_set.csv")
        self.labels = []
        # map labels to integers and store in a dictionary. Csv column 1 is "filename", column 2 is "label"

        self.label_map = {}
        with open(labels_dir, "r") as f:
            next(f)  # skip header
            for line in f:
                filename, label = line.strip().split(",")
                self.labels.append((filename, label))
                if label not in self.label_map:
                    self.label_map[label] = len(self.label_map)
        # convert labels to integers
        self.labels = [(fname, self.label_map[label]) for fname, label in self.labels]
        print("Label mapping:", self.label_map)

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx][1]
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [25]:
dataset = Butterflydataset(root_dir=path, transform=transform)
from torch.utils.data import random_split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")



Label mapping: {'SOUTHERN DOGFACE': 0, 'ADONIS': 1, 'BROWN SIPROETA': 2, 'MONARCH': 3, 'GREEN CELLED CATTLEHEART': 4, 'CAIRNS BIRDWING': 5, 'EASTERN DAPPLE WHITE': 6, 'RED POSTMAN': 7, 'MANGROVE SKIPPER': 8, 'BLACK HAIRSTREAK': 9, 'CABBAGE WHITE': 10, 'RED ADMIRAL': 11, 'PAINTED LADY': 12, 'PAPER KITE': 13, 'SOOTYWING': 14, 'PINE WHITE': 15, 'PEACOCK': 16, 'CHECQUERED SKIPPER': 17, 'JULIA': 18, 'COMMON WOOD-NYMPH': 19, 'BLUE MORPHO': 20, 'CLOUDED SULPHUR': 21, 'STRAITED QUEEN': 22, 'ORANGE OAKLEAF': 23, 'PURPLISH COPPER': 24, 'ATALA': 25, 'IPHICLUS SISTER': 26, 'DANAID EGGFLY': 27, 'LARGE MARBLE': 28, 'PIPEVINE SWALLOW': 29, 'BLUE SPOTTED CROW': 30, 'RED CRACKER': 31, 'QUESTION MARK': 32, 'CRIMSON PATCH': 33, 'BANDED PEACOCK': 34, 'SCARCE SWALLOW': 35, 'COPPER TAIL': 36, 'GREAT JAY': 37, 'INDRA SWALLOW': 38, 'VICEROY': 39, 'MALACHITE': 40, 'APPOLLO': 41, 'TWO BARRED FLASHER': 42, 'MOURNING CLOAK': 43, 'TROPICAL LEAFWING': 44, 'POPINJAY': 45, 'ORANGE TIP': 46, 'GOLD BANDED': 47, 'BECKER

In [26]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [32]:
# Define a simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=75):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 56 * 56)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
model = SimpleCNN(num_classes=75)
print(model)



SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=200704, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=75, bias=True)
)


In [33]:
#Training loop
import torch.optim as optim
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_dataloader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

#Evaluation on validation set
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Validation Accuracy: {100 * correct / total:.2f}%")



Epoch 1/10, Loss: 4.4501
Epoch 2/10, Loss: 4.3167
Epoch 3/10, Loss: 4.3151
Epoch 4/10, Loss: 4.2946
Epoch 5/10, Loss: 3.7584
Epoch 6/10, Loss: 1.7021
Epoch 7/10, Loss: 0.2920
Epoch 8/10, Loss: 0.0911
Epoch 9/10, Loss: 0.0479
Epoch 10/10, Loss: 0.0403
Validation Accuracy: 1.75%
